# exp_6 · 00 protocol — freeze, tests, G1
Freezes `PREREG.md` (SHA-256 + code-derived constants + SHA-256 of every reused file) before any detector
training or simulated-view scoring, records the write guard, runs the synthetic tests and computes G1 from
ATD + homography only (no frame). **Reads BAR-ATH ATD: never open this notebook executed.**

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp6_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_6')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp6_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log

if not C.PROTOCOL.exists():
    C.freeze_prereg()
    print('PREREG frozen now (before any detector training or simulated-view scoring)')
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

before = C.guard_record(C.RESULTS / 'write_guard_before.json')
print('write guard recorded:', {k: v['files'] for k, v in before['folders'].items()})
print('public check spent by exp_1 (never repeated):', C.public_check_spent())

PREREG frozen now (before any detector training or simulated-view scoring)
PREREG verified dca69fd7188c7ef8 | frozen 2026-11-10T14:57:58+00:00


write guard recorded: {'exp_0': 42630, 'exp_1': 215, 'exp_2': 312, 'exp_3': 407, 'exp_4': 638, 'exp_5': 247, 'data': 12, 'data_metric_repo': 1}
public check spent by exp_1 (never repeated): True


In [2]:
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests'],
                        cwd=ROOT, capture_output=True, text=True, env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'})
summary = result.stdout.strip().splitlines()[-1]
C1.write_json(C.RESULTS / 'tests.json', {'run_at_utc': now(), 'returncode': result.returncode, 'summary': summary})
print(summary)
if result.returncode:
    raise RuntimeError('exp_6 synthetic tests failed')

17 passed, 1 warning in 10.71s


In [3]:
import exp6_data as DA
inputs = DA.load_inputs()
summary = DA.input_summary(inputs)
C1.write_json(C.RESULTS / 'g1.json', {**summary['g1'], 'computed_at_utc': now()})
C1.write_json(C.RESULTS / 'inputs_summary.json', summary)
print('G1 (aggregate):', summary['g1'])
print('splits (aggregate):', summary['splits'])
del inputs

G1 (aggregate): {'visible_graded': 94214, 'inside_with_margin': 94029, 'share': 0.9980363852505997}
splits (aggregate): {'train': {'frames': 56093, 'runs': 91, 'labelled_frames': 55967}, 'val': {'frames': 19169, 'runs': 29, 'labelled_frames': 19169}, 'test': {'frames': 18952, 'runs': 29, 'labelled_frames': 18893}, 'hidden': {'frames': 64643, 'runs': 150, 'labelled_frames': 0}}
